In [1]:
!pip install -q xgboost scikit-image scikit-learn opencv-python-headless seaborn joblib

## 2. Imports & Config

In [2]:
import json
import warnings
from pathlib import Path
from collections import defaultdict

import cv2
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, precision_recall_curve,
    average_precision_score, roc_curve, auc
)

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')
TRAIN_CSV  = DATA_DIR / 'Train.csv'
TEST_CSV   = DATA_DIR / 'Test.csv'
TRAIN_DIR  = DATA_DIR
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE        = (64, 64)   
HOG_ORIENTATIONS    = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HSV_BINS        = (8, 8, 8)
USE_SIFT        = False      # set True to add BOW-SIFT features (slow)
SIFT_VOCAB_SIZE = 64

XGB_N_ESTIMATORS  = 500
XGB_MAX_DEPTH     = 6
XGB_LR            = 0.1
XGB_SUBSAMPLE     = 0.8
XGB_COLSAMPLE     = 0.8

CLASS_NAMES = [
    'Speed limit (20km/h)', 'Speed limit (30km/h)', 'Speed limit (50km/h)',
    'Speed limit (60km/h)', 'Speed limit (70km/h)', 'Speed limit (80km/h)',
    'End of speed limit (80km/h)', 'Speed limit (100km/h)', 'Speed limit (120km/h)',
    'No passing', 'No passing veh over 3.5 tons', 'Right-of-way at intersection',
    'Priority road', 'Yield', 'Stop', 'No vehicles', 'Veh > 3.5 tons prohibited',
    'No entry', 'General caution', 'Dangerous curve left', 'Dangerous curve right',
    'Double curve', 'Bumpy road', 'Slippery road', 'Road narrows on the right',
    'Road work', 'Traffic signals', 'Pedestrians', 'Children crossing',
    'Bicycles crossing', 'Beware of ice/snow', 'Wild animals crossing',
    'End speed + passing limits', 'Turn right ahead', 'Turn left ahead',
    'Ahead only', 'Go straight or right', 'Go straight or left',
    'Keep right', 'Keep left', 'Roundabout mandatory',
    'End of no passing', 'End no passing veh > 3.5 tons'
]

print('Config ready.')
print(f'Data dir exists: {DATA_DIR.exists()}')

Config ready.
Data dir exists: True


## 3. Feature Extraction

In [3]:
# ── features.py ────────────────────────────────────────────────────────────

def extract_hog(image, orientations=HOG_ORIENTATIONS,
                pixels_per_cell=HOG_PIXELS_PER_CELL,
                cells_per_block=HOG_CELLS_PER_BLOCK):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    features = hog(
        gray,
        orientations=orientations,
        pixels_per_cell=pixels_per_cell,
        cells_per_block=cells_per_block,
        feature_vector=True,
        block_norm='L2-Hys',
    )
    return features.astype(np.float32)


def extract_hsv_histogram(image, bins=HSV_BINS):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, list(bins), [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten().astype(np.float32)


def build_sift_vocabulary(images, vocab_size=SIFT_VOCAB_SIZE):
    sift = cv2.SIFT_create()
    bow_trainer = cv2.BOWKMeansTrainer(vocab_size)
    for img in images:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
        _, descriptors = sift.detectAndCompute(gray, None)
        if descriptors is not None:
            bow_trainer.add(descriptors)
    vocabulary = bow_trainer.cluster()
    matcher = cv2.BFMatcher(cv2.NORM_L2)
    bow_extractor = cv2.BOWImgDescriptorExtractor(sift, matcher)
    bow_extractor.setVocabulary(vocabulary)
    return vocabulary, bow_extractor


def extract_sift_bow(image, bow_extractor, detector=None):
    if detector is None:
        detector = cv2.SIFT_create()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    keypoints = detector.detect(gray)
    if not keypoints:
        return np.zeros(bow_extractor.getVocabularySize(), dtype=np.float32)
    desc = bow_extractor.compute(gray, keypoints)
    return desc.flatten().astype(np.float32)


def extract_features(image, use_sift=USE_SIFT, bow_extractor=None):
    parts = [extract_hog(image), extract_hsv_histogram(image)]
    if use_sift and bow_extractor is not None:
        parts.append(extract_sift_bow(image, bow_extractor))
    return np.concatenate(parts)


# Quick sanity-check
dummy = np.zeros((64, 64, 3), dtype=np.uint8)
print(f'Feature vector length: {extract_features(dummy).shape[0]}')

Feature vector length: 2276


## 4. Preprocessing

In [4]:
# ── preprocessing.py (inline) ──────────────────────────────────────────────

def prepare_crop(image, filter_method='gaussian', target_size=IMG_SIZE):
    """Resize + optional denoising."""
    img = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
    if filter_method == 'gaussian':
        img = cv2.GaussianBlur(img, (3, 3), 0)
    elif filter_method == 'bilateral':
        img = cv2.bilateralFilter(img, d=5, sigmaColor=75, sigmaSpace=75)
    elif filter_method == 'median':
        img = cv2.medianBlur(img, 3)
    return img


print('Preprocessing helpers ready.')

Preprocessing helpers ready.


In [5]:
def load_gtsrb_split(csv_path, base_dir, split_name='Train', max_samples=None):
    """
    Load GTSRB from a CSV file.
    Handles the Kaggle GTSRB layout where Path column contains e.g. 'Train/00000/00000_00000.png'
    """
    df = pd.read_csv(csv_path)
    print(f'[{split_name}] CSV shape: {df.shape}, columns: {list(df.columns)}')

    # Detect columns
    path_col  = next((c for c in ['Path', 'image_path', 'filename', 'Image'] if c in df.columns), None)
    class_col = next((c for c in ['ClassId', 'class', 'label', 'class_id']  if c in df.columns), None)
    roi_cols  = None
    for cset in [['Roi.X1','Roi.Y1','Roi.X2','Roi.Y2'], ['x1','y1','x2','y2'],
                 ['xmin','ymin','xmax','ymax'], ['x','y','w','h']]:
        if all(c in df.columns for c in cset):
            roi_cols = cset
            break

    assert path_col,  f'Cannot find image path column; got {list(df.columns)}'
    assert class_col, f'Cannot find class column; got {list(df.columns)}'

    if max_samples:
        df = df.sample(n=min(max_samples, len(df)), random_state=42).reset_index(drop=True)

    images, labels = [], []
    missing = 0

    for _, row in df.iterrows():
        img_path = base_dir / str(row[path_col])

        if not img_path.exists():
            # fallback: try without leading folder
            img_path = base_dir / Path(row[path_col]).name

        if not img_path.exists():
            missing += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            missing += 1
            continue

        # Crop to ROI if available
        if roi_cols:
            try:
                x1, y1, x2, y2 = [int(row[c]) for c in roi_cols]
                h, w = img.shape[:2]
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                if x2 > x1 and y2 > y1:
                    img = img[y1:y2, x1:x2]
            except Exception:
                pass

        processed = prepare_crop(img)
        images.append(processed)
        labels.append(int(row[class_col]))

    print(f'[{split_name}] Loaded {len(images)} images, missing={missing}')
    return images, np.array(labels)


print('Loading training data…')
train_images, train_labels = load_gtsrb_split(TRAIN_CSV, TRAIN_DIR, 'Train')

print('\nLoading test data…')
test_images, test_labels = load_gtsrb_split(TEST_CSV, TRAIN_DIR, 'Test')

print(f'\nTotal train: {len(train_images)} | Total test: {len(test_images)}')
print(f'Classes: {len(np.unique(train_labels))}')

Loading training data…
[Train] CSV shape: (39209, 8), columns: ['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId', 'Path']
[Train] Loaded 39209 images, missing=0

Loading test data…
[Test] CSV shape: (12630, 8), columns: ['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId', 'Path']
[Test] Loaded 12630 images, missing=0

Total train: 39209 | Total test: 12630
Classes: 43


In [6]:
def plot_class_distribution(labels, class_names=None, title='Class Distribution', save_path=None):
    unique, counts = np.unique(labels, return_counts=True)
    names = [class_names[u] if class_names else str(u) for u in unique]

    fig, ax = plt.subplots(figsize=(16, 5))
    bars = ax.bar(range(len(unique)), counts,
                  color=plt.cm.tab20(np.linspace(0, 1, len(unique))))
    ax.set_xticks(range(len(unique)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Sample Count')
    ax.set_title(title)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(cnt), ha='center', va='bottom', fontsize=6)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_class_distribution(
    train_labels,
    class_names=CLASS_NAMES,
    title='GTSRB Training Set — Class Distribution',
    save_path=OUTPUT_DIR / 'class_distribution.png'
)

Saved → /kaggle/working/class_distribution.png


In [7]:
bow_extractor = None

if USE_SIFT:
    print('Building SIFT vocabulary…')
    _, bow_extractor = build_sift_vocabulary(train_images, vocab_size=SIFT_VOCAB_SIZE)

print('Extracting training features…')
X_train_raw = np.array(
    [extract_features(img, bow_extractor=bow_extractor) for img in train_images],
    dtype=np.float32
)
y_train_raw = train_labels

print('Extracting test features…')
X_test_raw = np.array(
    [extract_features(img, bow_extractor=bow_extractor) for img in test_images],
    dtype=np.float32
)
y_test_raw = test_labels

print(f'Feature shape — train: {X_train_raw.shape} | test: {X_test_raw.shape}')

Extracting training features…
Extracting test features…
Feature shape — train: (39209, 2276) | test: (12630, 2276)


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

# Hold-out validation split from training data
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train_raw,
    test_size=0.15, random_state=42, stratify=y_train_raw
)

num_classes = len(np.unique(y_train_raw))
print(f'Train: {X_tr.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test_scaled.shape[0]}')
print(f'Num classes: {num_classes}')

Train: 33327 | Val: 5882 | Test: 12630
Num classes: 43


In [9]:
model = xgb.XGBClassifier(
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LR,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    objective='multi:softprob',
    num_class=num_classes,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    tree_method='hist',
    device='cuda',         
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f'\nBest iteration: {model.best_iteration}')

[0]	validation_0-mlogloss:2.32941
[50]	validation_0-mlogloss:0.16070
[100]	validation_0-mlogloss:0.08568
[150]	validation_0-mlogloss:0.07036
[200]	validation_0-mlogloss:0.06471
[250]	validation_0-mlogloss:0.06190
[300]	validation_0-mlogloss:0.06052
[350]	validation_0-mlogloss:0.05958
[400]	validation_0-mlogloss:0.05888
[450]	validation_0-mlogloss:0.05829
[499]	validation_0-mlogloss:0.05786

Best iteration: 498


## 10. Predict & Core Metrics

In [10]:
y_pred       = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)   # shape (N, 43)

acc = accuracy_score(y_test_raw, y_pred)
f1  = f1_score(y_test_raw, y_pred, average='weighted')
f1_macro  = f1_score(y_test_raw, y_pred, average='macro')
f1_per_class = f1_score(y_test_raw, y_pred, average=None)

print(f'Accuracy          : {acc:.4f}')
print(f'F1 Weighted       : {f1:.4f}')
print(f'F1 Macro          : {f1_macro:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_raw, y_pred, target_names=CLASS_NAMES))

# Save metrics
metrics = {
    'accuracy': float(acc),
    'f1_weighted': float(f1),
    'f1_macro': float(f1_macro),
    'best_iteration': int(model.best_iteration),
    'num_classes': num_classes,
    'feature_dim': X_tr.shape[1],
}
with open(OUTPUT_DIR / 'xgb_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('\nMetrics saved.')

Accuracy          : 0.9455
F1 Weighted       : 0.9446
F1 Macro          : 0.9248

Classification Report:
                               precision    recall  f1-score   support

         Speed limit (20km/h)       0.96      0.78      0.86        60
         Speed limit (30km/h)       0.95      0.94      0.94       720
         Speed limit (50km/h)       0.92      0.99      0.95       750
         Speed limit (60km/h)       0.99      0.88      0.93       450
         Speed limit (70km/h)       0.94      0.97      0.96       660
         Speed limit (80km/h)       0.84      0.93      0.89       630
  End of speed limit (80km/h)       0.87      0.69      0.77       150
        Speed limit (100km/h)       0.95      0.96      0.95       450
        Speed limit (120km/h)       0.94      0.87      0.90       450
                   No passing       0.95      0.98      0.96       480
 No passing veh over 3.5 tons       0.95      0.98      0.97       660
 Right-of-way at intersection       0.94  

## 11. Confusion Matrix

In [11]:
def plot_confusion_matrix(y_true, y_pred, class_names, save_path=None, figsize_scale=0.45):
    cm = confusion_matrix(y_true, y_pred)
    n  = len(class_names)
    fs = max(10, int(n * figsize_scale))

    fig, ax = plt.subplots(figsize=(fs, fs - 1))
    sns.heatmap(
        cm, annot=(n <= 20), fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.3 if n <= 20 else 0,
    )
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)
    ax.set_title('Confusion Matrix — XGBoost on GTSRB', fontsize=13)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_confusion_matrix(
    y_test_raw, y_pred,
    class_names=CLASS_NAMES,
    save_path=OUTPUT_DIR / 'confusion_matrix.png'
)

Saved → /kaggle/working/confusion_matrix.png


## 12. Precision-Recall Curves

In [12]:
def plot_precision_recall(y_true, y_scores, class_names, save_path=None, max_classes=15):
    """
    Plots one PR curve per class (limited to max_classes for readability).
    Also plots the micro-average.
    """
    n_classes = y_scores.shape[1]
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))

    # Compute micro-average
    prec_micro, rec_micro, _ = precision_recall_curve(y_bin.ravel(), y_scores.ravel())
    ap_micro = average_precision_score(y_bin, y_scores, average='micro')

    # Per-class APs — pick top max_classes by AP for legibility
    per_class_ap = [
        average_precision_score(y_bin[:, i], y_scores[:, i])
        for i in range(n_classes)
    ]
    top_idx = np.argsort(per_class_ap)[::-1][:max_classes]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: micro-average
    axes[0].plot(rec_micro, prec_micro, color='steelblue', lw=2,
                 label=f'Micro-avg (mAP={ap_micro:.3f})')
    axes[0].fill_between(rec_micro, prec_micro, alpha=0.15, color='steelblue')
    axes[0].set_xlabel('Recall')
    axes[0].set_ylabel('Precision')
    axes[0].set_title('Micro-Average Precision-Recall')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Right: per-class (top N)
    cmap = plt.cm.tab20(np.linspace(0, 1, len(top_idx)))
    for color, i in zip(cmap, top_idx):
        prec_i, rec_i, _ = precision_recall_curve(y_bin[:, i], y_scores[:, i])
        ap_i = per_class_ap[i]
        axes[1].plot(rec_i, prec_i, color=color, lw=1.5,
                     label=f'{class_names[i][:22]} ({ap_i:.2f})')

    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'Per-Class PR — Top {max_classes} by AP')
    axes[1].legend(fontsize=7, loc='lower left')
    axes[1].grid(alpha=0.3)

    plt.suptitle('Precision-Recall Curves — XGBoost on GTSRB', fontsize=13, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()

    return per_class_ap, ap_micro


per_class_ap, ap_micro = plot_precision_recall(
    y_test_raw, y_pred_proba,
    class_names=CLASS_NAMES,
    save_path=OUTPUT_DIR / 'precision_recall_curves.png'
)
print(f'\nMicro-avg AP: {ap_micro:.4f}')

Saved → /kaggle/working/precision_recall_curves.png

Micro-avg AP: 0.9851


## 13. Per-Class F1 Bar Chart

In [13]:
def plot_per_class_f1(f1_per_class, class_names, save_path=None):
    order = np.argsort(f1_per_class)
    colors = plt.cm.RdYlGn(f1_per_class[order])

    fig, ax = plt.subplots(figsize=(10, 14))
    bars = ax.barh(range(len(order)), f1_per_class[order], color=colors)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([class_names[i][:35] for i in order], fontsize=8)
    ax.set_xlabel('F1 Score')
    ax.set_title('Per-Class F1 Score — XGBoost on GTSRB')
    ax.axvline(x=np.mean(f1_per_class), color='navy', linestyle='--',
               linewidth=1.5, label=f'Mean F1={np.mean(f1_per_class):.3f}')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_per_class_f1(
    f1_per_class, CLASS_NAMES,
    save_path=OUTPUT_DIR / 'per_class_f1.png'
)

Saved → /kaggle/working/per_class_f1.png


## 14. Feature Importance

In [14]:
def plot_feature_importance(model, top_n=30, save_path=None):
    importance = model.feature_importances_
    indices = np.argsort(importance)[::-1][:top_n]

    # Label: HOG vs HSV block
    hog_dim = (IMG_SIZE[0]//8 - 1) * (IMG_SIZE[1]//8 - 1) * 4 * 9
    def feat_name(i):
        if i < hog_dim:
            return f'HOG_{i}'
        else:
            return f'HSV_{i - hog_dim}'

    names  = [feat_name(i) for i in indices]
    values = importance[indices]
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(values)))

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.3)))
    ax.barh(range(len(values)), values[::-1], color=colors[::-1])
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(names[::-1], fontsize=9)
    ax.set_xlabel('Feature Importance (Gain)')
    ax.set_title(f'XGBoost Top {top_n} Feature Importances')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_feature_importance(
    model, top_n=30,
    save_path=OUTPUT_DIR / 'feature_importance.png'
)

Saved → /kaggle/working/feature_importance.png


## 15. Learning Curve (Validation Loss)

In [15]:
def plot_learning_curve(model, save_path=None):
    results = model.evals_result()
    val_key = list(results.keys())[0]
    metric_key = list(results[val_key].keys())[0]
    losses = results[val_key][metric_key]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(losses, color='steelblue', lw=2, label='Validation mlogloss')
    ax.axvline(x=model.best_iteration, color='tomato', linestyle='--',
               label=f'Best iter={model.best_iteration}')
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel('Log Loss')
    ax.set_title('XGBoost Training — Validation Log Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_learning_curve(
    model,
    save_path=OUTPUT_DIR / 'learning_curve.png'
)

Saved → /kaggle/working/learning_curve.png


## 16. Summary Dashboard

In [16]:
def plot_summary_dashboard(y_true, y_pred, f1_per_class, per_class_ap,
                           class_names, acc, f1_weighted, save_path=None):
    fig = plt.figure(figsize=(18, 10))
    gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.4)

    # 1. Metric summary
    ax0 = fig.add_subplot(gs[0, 0])
    metric_names  = ['Accuracy', 'F1 Weighted', 'F1 Macro', 'mAP']
    metric_values = [acc, f1_weighted, float(np.mean(f1_per_class)), float(np.mean(per_class_ap))]
    bars = ax0.bar(metric_names, metric_values,
                   color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    ax0.set_ylim(0, 1.1)
    ax0.set_title('Overall Metrics')
    for b, v in zip(bars, metric_values):
        ax0.text(b.get_x() + b.get_width()/2, v + 0.01,
                 f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax0.grid(axis='y', alpha=0.3)

    # 2. F1 histogram
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.hist(f1_per_class, bins=20, color='steelblue', edgecolor='white')
    ax1.axvline(np.mean(f1_per_class), color='tomato', linestyle='--',
                label=f'Mean={np.mean(f1_per_class):.3f}')
    ax1.set_xlabel('F1 Score')
    ax1.set_ylabel('# Classes')
    ax1.set_title('F1 Distribution Across Classes')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # 3. AP histogram
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(per_class_ap, bins=20, color='seagreen', edgecolor='white')
    ax2.axvline(np.mean(per_class_ap), color='tomato', linestyle='--',
                label=f'mAP={np.mean(per_class_ap):.3f}')
    ax2.set_xlabel('Average Precision')
    ax2.set_ylabel('# Classes')
    ax2.set_title('AP Distribution Across Classes')
    ax2.legend()
    ax2.grid(alpha=0.3)

    # 4. Top-10 best F1 classes
    ax3 = fig.add_subplot(gs[1, 0])
    top10 = np.argsort(f1_per_class)[::-1][:10]
    ax3.barh(range(10), f1_per_class[top10][::-1], color='seagreen')
    ax3.set_yticks(range(10))
    ax3.set_yticklabels([class_names[i][:25] for i in top10[::-1]], fontsize=8)
    ax3.set_xlabel('F1 Score')
    ax3.set_title('Top 10 Classes by F1')
    ax3.grid(axis='x', alpha=0.3)

    # 5. Bottom-10 worst F1 classes
    ax4 = fig.add_subplot(gs[1, 1])
    bot10 = np.argsort(f1_per_class)[:10]
    ax4.barh(range(10), f1_per_class[bot10][::-1], color='tomato')
    ax4.set_yticks(range(10))
    ax4.set_yticklabels([class_names[i][:25] for i in bot10[::-1]], fontsize=8)
    ax4.set_xlabel('F1 Score')
    ax4.set_title('Bottom 10 Classes by F1')
    ax4.grid(axis='x', alpha=0.3)

    # 6. F1 vs AP scatter
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.scatter(f1_per_class, per_class_ap, alpha=0.6, color='steelblue', edgecolors='white')
    ax5.set_xlabel('F1 Score')
    ax5.set_ylabel('Average Precision')
    ax5.set_title('F1 vs AP per Class')
    ax5.grid(alpha=0.3)
    # Annotate outlier classes
    for i in np.argsort(f1_per_class)[:3]:
        ax5.annotate(class_names[i][:15], (f1_per_class[i], per_class_ap[i]),
                     fontsize=7, color='tomato')

    fig.suptitle('XGBoost on GTSRB — Evaluation Summary', fontsize=15, fontweight='bold')
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_summary_dashboard(
    y_test_raw, y_pred,
    f1_per_class=f1_per_class,
    per_class_ap=np.array(per_class_ap),
    class_names=CLASS_NAMES,
    acc=acc, f1_weighted=f1,
    save_path=OUTPUT_DIR / 'summary_dashboard.png'
)

Saved → /kaggle/working/summary_dashboard.png


## 17. Save Model Artifacts

In [17]:
joblib.dump(model,   OUTPUT_DIR / 'xgb_model.pkl')
joblib.dump(scaler,  OUTPUT_DIR / 'xgb_scaler.pkl')

np.save(OUTPUT_DIR / 'xgb_test_labels.npy', y_test_raw)
np.save(OUTPUT_DIR / 'xgb_test_preds.npy',  y_pred)
np.save(OUTPUT_DIR / 'xgb_test_proba.npy',  y_pred_proba)

if bow_extractor is not None:
    joblib.dump(bow_extractor, OUTPUT_DIR / 'bow_extractor.pkl')

print('All artifacts saved to', OUTPUT_DIR)
print(list(OUTPUT_DIR.glob('*')))

All artifacts saved to /kaggle/working
[PosixPath('/kaggle/working/.virtual_documents'), PosixPath('/kaggle/working/xgb_metrics.json'), PosixPath('/kaggle/working/xgb_model.pkl'), PosixPath('/kaggle/working/xgb_test_proba.npy'), PosixPath('/kaggle/working/confusion_matrix.png'), PosixPath('/kaggle/working/xgb_test_preds.npy'), PosixPath('/kaggle/working/xgb_scaler.pkl'), PosixPath('/kaggle/working/learning_curve.png'), PosixPath('/kaggle/working/feature_importance.png'), PosixPath('/kaggle/working/xgb_test_labels.npy'), PosixPath('/kaggle/working/precision_recall_curves.png'), PosixPath('/kaggle/working/per_class_f1.png'), PosixPath('/kaggle/working/summary_dashboard.png'), PosixPath('/kaggle/working/class_distribution.png')]


## 18. Sample Predictions Grid

In [18]:
def plot_sample_predictions(images, y_true, y_pred, class_names,
                             n=16, cols=4, save_path=None):
    """Show a random grid of test images with True vs Predicted labels."""
    idx = np.random.choice(len(images), size=n, replace=False)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).flatten()

    for ax, i in zip(axes, idx):
        rgb = cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB)
        ax.imshow(rgb)
        t = class_names[y_true[i]]
        p = class_names[y_pred[i]]
        correct = (y_true[i] == y_pred[i])
        ax.set_title(f'T: {t[:18]}\nP: {p[:18]}',
                     fontsize=7,
                     color='green' if correct else 'red')
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle('Sample Predictions (green=correct, red=wrong)', fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()
    plt.close()


plot_sample_predictions(
    test_images, y_test_raw, y_pred,
    class_names=CLASS_NAMES,
    n=16, cols=4,
    save_path=OUTPUT_DIR / 'sample_predictions.png'
)

Saved → /kaggle/working/sample_predictions.png
